# Game recommendations with implicit-feedback ALS (prototype)

Our outside-the-course method: matrix factorisation for implicit feedback (Hu, Koren & Volinsky, 2008), solved with alternating least squares. Each user and game gets a vector of `k` hidden factors, and their dot product scores how likely the user is to play the game. Playtime is used as *confidence* (`c = 1 + α·log(1 + minutes)`), not as a rating.

This prototype compares our own ALS implementation (`recommender/als.py`) with a popularity baseline on held-out games, and cross-checks it against the `implicit` library. Hyperparameters are only lightly tuned.

## Data

Load the user inventories and keep games owned by at least 50 users (the same threshold as the Apriori/PCY baskets) and users with at least 5 played games. Owned-but-unplayed games count as not played.

In [1]:
import time

import numpy as np
import pandas as pd

from recommender import als, data, evaluate

user_items = data.load_user_items()
inter = data.build_interactions(user_items)

n_users, n_games = inter.playtime.shape
print(f"{n_users:,} users x {n_games:,} games, {inter.playtime.nnz:,} played pairs "
      f"({inter.playtime.nnz / (n_users * n_games):.2%} of the matrix filled)")

59,732 users x 5,198 games, 3,184,647 played pairs (1.03% of the matrix filled)


## Train / validation / test split

For every user, 20% of their played games are hidden as the **test** set. From the rest, another 20% are hidden as a **validation** set used only to choose α. Models are scored on how many hidden games appear in their top-10 recommendations.

In [2]:
train_full, test = data.split_per_user(inter.playtime, test_frac=0.2, seed=0)
train, valid = data.split_per_user(train_full, test_frac=0.2, seed=1)

print(f"train {train.nnz:,}  valid {valid.nnz:,}  test {test.nnz:,}")

train 2,038,529  valid 509,535  test 636,583


## Choosing α on the validation set

α controls how much extra trust long playtimes get. Factors (32), regularisation (1.0) and sweeps (15) are fixed for the prototype.

In [3]:
FACTORS, REG, ITERATIONS = 32, 1.0, 15

rows = []
for alpha in [0.3, 1.0, 3.0, 10.0]:
    model = als.fit(als.confidence(train, alpha), FACTORS, REG, ITERATIONS)
    recs = evaluate.top_k(evaluate.als_scorer(model), train)
    rows.append({"alpha": alpha, **evaluate.ranking_metrics(recs, valid)})

alpha_results = pd.DataFrame(rows).drop(columns="users")
best_alpha = alpha_results.loc[alpha_results["ndcg@10"].idxmax(), "alpha"]
display(alpha_results)
print(f"Best alpha on validation: {best_alpha}")

,alpha,precision@10,recall@10,ndcg@10
0,0.3,0.105538,0.163235,0.164519
1,1.0,0.120749,0.200505,0.194486
2,3.0,0.123960,0.218986,0.202864
3,10.0,0.099145,0.196770,0.167651


Best alpha on validation: 3.0


## Test results: popularity vs. ALS

Retrain on train + validation with the chosen α, then score the test games. *Coverage* is the share of all games that are recommended to at least one user.

In [4]:
start = time.perf_counter()
model = als.fit(als.confidence(train_full, best_alpha), FACTORS, REG, ITERATIONS)
fit_seconds = time.perf_counter() - start

candidates = {
    "Popularity": evaluate.popularity_scorer(train_full),
    "ALS (ours)": evaluate.als_scorer(model),
}

results = []
for name, scorer in candidates.items():
    recs = evaluate.top_k(scorer, train_full)
    results.append({
        "model": name,
        **evaluate.ranking_metrics(recs, test),
        "coverage": evaluate.catalogue_coverage(recs, n_games),
    })

print(f"ALS fit: {fit_seconds:.1f} s for {ITERATIONS} sweeps")
display(pd.DataFrame(results).drop(columns="users").set_index("model").round(3))

ALS fit: 2.1 s for 15 sweeps


,precision@10,recall@10,ndcg@10,coverage
model,,,,
Popularity,0.135,0.180,0.220,0.010
ALS (ours),0.180,0.244,0.269,0.184


## Cross-check against the `implicit` library

`implicit` implements the same paper. With the same confidence values and settings, its scores should be close to ours. We use its exact (Cholesky) solver, not its default conjugate-gradient approximation.

In [5]:
from implicit.cpu.als import AlternatingLeastSquares

log_playtime = train_full.copy()
log_playtime.data = np.log1p(log_playtime.data)  # implicit uses c = 1 + alpha * value

lib = AlternatingLeastSquares(
    factors=FACTORS, regularization=REG, alpha=best_alpha, iterations=ITERATIONS,
    use_cg=False, random_state=0,
)
start = time.perf_counter()
lib.fit(log_playtime, show_progress=False)
lib_seconds = time.perf_counter() - start

lib_model = als.ALSModel(np.asarray(lib.user_factors), np.asarray(lib.item_factors))
lib_recs = evaluate.top_k(evaluate.als_scorer(lib_model), train_full)
print(f"implicit fit: {lib_seconds:.1f} s")
display(pd.Series(evaluate.ranking_metrics(lib_recs, test)).drop("users").round(3))

implicit fit: 15.1 s


precision@10    0.181
recall@10       0.243
ndcg@10         0.269
dtype: float64

## Similar games

Games with similar vectors are played by the same kind of users. If the factors capture taste, the neighbours of a well-known game should make sense.

In [6]:
name_to_index = {inter.item_names[item_id]: i for i, item_id in enumerate(inter.item_ids)}

def show_similar(title, n=8):
    top, sims = als.similar_items(model, name_to_index[title], n)
    return pd.DataFrame({
        f"Similar to {title}": [inter.item_names[inter.item_ids[i]] for i in top],
        "cosine": sims[top].round(2),
    })

for title in ["Counter-Strike: Global Offensive", "Sid Meier's Civilization V", "Euro Truck Simulator 2", "Dark Souls: Prepare to Die Edition"]:
    if title in name_to_index:
        display(show_similar(title))

,Similar to Counter-Strike: Global Offensive,cosine
0,Garry's Mod,0.99
1,Left 4 Dead 2,0.98
2,Terraria,0.95
3,PAYDAY 2,0.94
4,Unturned,0.93
5,Portal 2,0.89
6,Counter-Strike: Source,0.87
7,Warframe,0.84


,Similar to Sid Meier's Civilization V,cosine
0,Age of Empires II: HD Edition,0.71
1,The Elder Scrolls V: Skyrim,0.65
2,FTL: Faster Than Light,0.61
3,XCOM: Enemy Unknown,0.61
4,Borderlands 2,0.59
5,Sid Meier's Civilization: Beyond Earth,0.58
6,Counter-Strike: Source,0.55
7,Portal,0.54


,Similar to Euro Truck Simulator 2,cosine
0,Bus Driver,0.85
1,Microsoft Flight Simulator X: Steam Edition,0.82
2,OMSI 2,0.78
3,Scania Truck Driving Simulator,0.77
4,American Truck Simulator,0.75
5,Farming Simulator 15,0.74
6,Euro Truck Simulator,0.74
7,Car Mechanic Simulator 2014,0.73


,Similar to Dark Souls: Prepare to Die Edition,cosine
0,DARK SOULS™ II,0.78
1,DARK SOULS™ II: Scholar of the First Sin,0.74
2,DARK SOULS III,0.71
3,Fallout 3 - Game of the Year Edition,0.68
4,The Witcher 2: Assassins of Kings Enhanced Edi...,0.66
5,The Witcher: Enhanced Edition,0.65
6,The Elder Scrolls IV: Oblivion,0.65
7,Fallout: New Vegas,0.63


## Notes for the next iteration

- The neighbours of very popular games are mostly other very popular games: one factor seems to capture overall popularity. More factors, stronger regularisation or down-weighting popular games (e.g. BM25) may help.
- Tune factors and regularisation on the validation split, not only α.
- Add the course-method comparisons: recommendations from the Apriori/PCY association rules, and item–item similarity (optionally with MinHash/LSH).
- Cluster the game vectors with k-means and compare the clusters with Steam tags.